# SHL Hiring Assessment 2026 — Spoken Grammar Scoring Engine
**Goal:** predict a continuous MOS grammar score (0–5) from a 45–60 s audio clip.

## Approach (Report)
1. **Speech → text (Whisper-large-v3, English).** Grammar is a property of *what* was said, so we transcribe each clip. We also extract **encoder hidden-state embeddings** (mean+std pooled across 7 layers) — these capture fluency, hesitation and learner-like speech patterns far beyond what the transcript alone reveals.
2. **wav2vec2-base encoder embeddings.** A second acoustic representation trained on 960h of LibriSpeech. Captures complementary pronunciation and rhythm information.
3. **Text features from the transcript.** Word count, speech rate (WPM), lexical diversity (TTR), sentence lengths, disfluencies (um/uh), word repetitions, **GPT-2 perplexity** (ungrammatical text → higher perplexity), and **LanguageTool grammar error counts**.
4. **Audio features from librosa.** MFCCs, pitch statistics, energy (RMS), zero-crossing rate, tempo, spectral centroid, and **pause ratio**.
5. **Models.** Ridge, SVR, Kernel Ridge, ElasticNet, and LightGBM on standardised features, then a **Nelder-Mead optimised stacking blend** learned from out-of-fold predictions.
6. **Evaluation.** Repeated 5-fold CV (RMSE, Pearson), plus training RMSE (required). Predictions clipped to [0, 5].

> Enable **GPU T4** and **Internet** in the Kaggle notebook settings.

In [ ]:
!pip install -q language_tool_python lightgbm

In [ ]:
import os, re, glob, warnings, pickle, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, torch, librosa
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import RepeatedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from scipy.optimize import minimize, nnls
import lightgbm as lgb

warnings.filterwarnings('ignore')
WORK = Path('/kaggle/working')

ROOT = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
if not os.path.exists(ROOT):
    candidates = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if candidates:
        ROOT = os.path.dirname(candidates[0])
    else:
        raise FileNotFoundError('Cannot find train.csv')

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000; SEED = 42
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
print(f'ROOT: {ROOT}  |  Device: {DEV}')
print(os.listdir(ROOT))

## 1. Data Loading & EDA

In [ ]:
train = pd.read_csv(f'{ROOT}/train.csv')
test  = pd.read_csv(f'{ROOT}/test.csv')
sub   = pd.read_csv(f'{ROOT}/sample_submission.csv')
print(train.head(), '\n'); print(test.head(), '\n'); print(sub.head())

FCOL = train.columns[0]
LCOL = [c for c in train.columns if c != FCOL and pd.api.types.is_numeric_dtype(train[c])][0]
print(f'\nFile col: {FCOL} | Label col: {LCOL}')
print(f'Train: {len(train)} | Test: {len(test)}')

def apath(split, f):
    f = str(f); f = f if f.endswith('.wav') else f + '.wav'
    return f'{ROOT}/{split}/{f}'

train['path'] = [apath('train', f) for f in train[FCOL]]
test['path']  = [apath('test', f) for f in test[FCOL]]
y = train[LCOL].values.astype(float)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(y, bins=20, kde=True, ax=ax[0], color='#4C72B0')
ax[0].set_title('Train Label Distribution'); ax[0].set_xlabel('Grammar Score')
sns.countplot(x=np.round(y*2)/2, ax=ax[1], color='#55A868')
ax[1].set_title('Rounded Labels (0.5 bins)'); ax[1].set_xlabel('Grammar Score')
sns.boxplot(y=y, ax=ax[2], color='#C44E52')
ax[2].set_title('Label Box Plot'); ax[2].set_ylabel('Grammar Score')
plt.tight_layout(); plt.show()
print(train[LCOL].describe())
print(f'\nBaseline RMSE (predict mean): {np.std(y):.4f}')

## 2. Tower A: Whisper-large-v3 Encoder Embeddings + Transcripts (Cached)
We use `whisper-large-v3` (1.55B params, 32 encoder layers) instead of `whisper-small` for substantially richer acoustic-linguistic representations. We pool 7 layers to capture phonetic through syntactic information.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

WNAME = 'openai/whisper-large-v3'
WHISPER_CACHE = str(WORK / 'whisper_large_cache.pkl')
LAYERS = [8, 12, 16, 20, 24, 28, 32]

if os.path.exists(WHISPER_CACHE):
    print('Loading cached Whisper features...')
    cache = pickle.load(open(WHISPER_CACHE, 'rb'))
else:
    print(f'Loading {WNAME}...')
    proc = WhisperProcessor.from_pretrained(WNAME)
    wh = WhisperForConditionalGeneration.from_pretrained(
        WNAME, torch_dtype=torch.float16 if DEV=='cuda' else torch.float32
    ).to(DEV).eval()

    def load_chunks(p):
        a, _ = librosa.load(p, sr=SR, mono=True)
        a = librosa.util.normalize(a) if np.abs(a).max() > 0 else a
        n = 30 * SR
        chunks = [a[i:i+n] for i in range(0, max(len(a),1), n)]
        if len(chunks) > 1 and len(chunks[-1]) < 2*SR:
            chunks = chunks[:-1]
        return chunks, len(a)/SR

    @torch.no_grad()
    def process(p):
        chunks, dur = load_chunks(p)
        inp = proc.feature_extractor(chunks, sampling_rate=SR, return_tensors='pt'
              ).input_features.to(DEV, wh.dtype)
        enc = wh.model.encoder(inp, output_hidden_states=True).hidden_states
        emb = []
        for L in LAYERS:
            if L <= len(enc) - 1:
                h = enc[L].float()
                emb.append(torch.cat([h.mean(1), h.std(1)], -1))
        emb = torch.cat(emb, -1).mean(0).cpu().numpy()
        ids = wh.generate(inp, language='en', task='transcribe', max_new_tokens=200, num_beams=1)
        txt = ' '.join(t.strip() for t in proc.batch_decode(ids, skip_special_tokens=True))
        return emb, txt, dur

    cache = {}
    for label, df in [('train', train), ('test', test)]:
        print(f'Processing {label}...')
        for p in tqdm(df.path):
            if p not in cache:
                cache[p] = process(p)
    pickle.dump(cache, open(WHISPER_CACHE, 'wb'))
    del wh, proc; torch.cuda.empty_cache()
    print('Whisper model unloaded to free GPU memory.')

E_tr = np.stack([cache[p][0] for p in train.path])
E_te = np.stack([cache[p][0] for p in test.path])
train['text'] = [cache[p][1] for p in train.path]
test['text']  = [cache[p][1] for p in test.path]
train['dur']  = [cache[p][2] for p in train.path]
test['dur']   = [cache[p][2] for p in test.path]
print(f'\n Whisper encoder embeddings: {E_tr.shape}')
print(train[['text', LCOL]].head(5).to_string())

## 3. Tower B: wav2vec2-base Encoder Embeddings (Cached)
A complementary acoustic representation from a model trained purely on speech audio. Captures pronunciation quality and rhythm.

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

W2V_CACHE = str(WORK / 'wav2vec2_cache.npy')

if os.path.exists(W2V_CACHE):
    print('Loading cached wav2vec2 features...')
    w2v_all = np.load(W2V_CACHE)
else:
    print('Loading wav2vec2-base...')
    w2v_ext = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')
    w2v_model = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(DEV).eval()

    @torch.no_grad()
    def w2v_embed(path):
        wav, _ = librosa.load(str(path), sr=16000, mono=True)
        chunk_len = 30 * 16000
        embs = []
        for i in range(0, max(len(wav),1), chunk_len):
            chunk = wav[i:i+chunk_len]
            if len(chunk) < 1600: continue
            inp = w2v_ext(chunk, sampling_rate=16000, return_tensors='pt').input_values.to(DEV)
            h = w2v_model(inp).last_hidden_state.float()
            embs.append(torch.cat([h.mean(1), h.std(1)], -1).squeeze().cpu().numpy())
        return np.mean(embs, axis=0) if embs else np.zeros(768*2)

    all_paths = list(train.path) + list(test.path)
    w2v_all = np.stack([w2v_embed(p) for p in tqdm(all_paths, desc='wav2vec2')])
    np.save(W2V_CACHE, w2v_all)
    del w2v_model, w2v_ext; torch.cuda.empty_cache()

W_tr = w2v_all[:len(train)]; W_te = w2v_all[len(train):]
print(f'\n wav2vec2 embeddings: {W_tr.shape}')

## 4. Tower C: Text Features (Fluency + GPT-2 Perplexity + LanguageTool)
GPT-2 perplexity serves as a grammar proxy: ungrammatical text receives higher perplexity. We also count grammar errors from LanguageTool.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import language_tool_python

TEXT_CACHE = str(WORK / 'text_features.npy')
TEXT_COLS_CACHE = str(WORK / 'text_cols.pkl')

if os.path.exists(TEXT_CACHE):
    print('Loading cached text features...')
    T_all = np.load(TEXT_CACHE)
    T_cols = pickle.load(open(TEXT_COLS_CACHE, 'rb')) if os.path.exists(TEXT_COLS_CACHE) else None
else:
    print('Loading GPT-2...')
    tok = AutoTokenizer.from_pretrained('gpt2')
    lm = AutoModelForCausalLM.from_pretrained('gpt2').to(DEV).eval()
    tok.pad_token = tok.eos_token

    @torch.no_grad()
    def lm_loss(s):
        ids = tok(s, return_tensors='pt', truncation=True, max_length=256).input_ids.to(DEV)
        if ids.shape[1] < 2: return np.nan
        return lm(ids, labels=ids).loss.item()

    print('Loading LanguageTool...')
    lt = language_tool_python.LanguageTool('en-US')
    DISFL = r'\b(um+|uh+|er+|ah+|hmm+|erm)\b'

    def text_feats(t, dur):
        t = str(t)
        w = re.findall(r"[A-Za-z']+", t.lower()); n = len(w) + 1e-6
        sents = [s.strip() for s in re.split(r'[.!?]+', t) if s.strip()]
        sl = [len(s.split()) for s in sents] or [0]
        sloss = [lm_loss(s) for s in sents if len(s.split()) >= 3]
        sloss = [x for x in sloss if x == x] or [np.nan]
        bi = list(zip(w, w[1:]))
        matches = lt.check(t); n_lt = len(matches)
        gram_e = spell_e = 0
        for m in matches:
            rs = str(getattr(m, 'rule_id', getattr(m, 'ruleId', getattr(m, 'category', '')))).upper()
            if any(k in rs for k in ['GRAMMAR','TYPO','SYNTAX']): gram_e += 1
            if any(k in rs for k in ['SPELL','MORFOLOGIK']): spell_e += 1
        return dict(
            n_words=len(w), wpm=len(w)/dur*60, ttr=len(set(w))/n,
            avg_wlen=np.mean([len(x) for x in w]) if w else 0,
            n_sents=len(sents), mean_sl=np.mean(sl), max_sl=np.max(sl), std_sl=np.std(sl),
            disfl=len(re.findall(DISFL, t.lower()))/n,
            rep_bigram=1-len(set(bi))/(len(bi)+1e-6) if bi else 0,
            rep_word=np.mean([a==b for a,b in zip(w,w[1:])]) if len(w)>1 else 0,
            ppl_all=lm_loss(t), ppl_sent_mean=np.nanmean(sloss),
            ppl_sent_max=np.nanmax(sloss), ppl_sent_std=np.nanstd(sloss),
            comma_rate=t.count(',')/n, i_rate=w.count('i')/n,
            func_rate=sum(x in {'the','a','an','is','are','was','were','to','of','in','and','that','it'} for x in w)/n,
            lt_errors=n_lt/n, lt_gram=gram_e/n, lt_spell=spell_e/n,
            long_words=sum(1 for x in w if len(x)>6)/n,
            short_sents=sum(1 for s in sl if s<5)/max(len(sl),1),
            chars_per_word=len(t.replace(' ',''))/n,
        )

    print('Extracting text features (train)...')
    T_tr_df = pd.DataFrame([text_feats(t,d) for t,d in tqdm(zip(train.text, train.dur), total=len(train))])
    print('Extracting text features (test)...')
    T_te_df = pd.DataFrame([text_feats(t,d) for t,d in tqdm(zip(test.text, test.dur), total=len(test))])
    med = T_tr_df.median(); T_tr_df = T_tr_df.fillna(med); T_te_df = T_te_df.fillna(med)
    T_cols = list(T_tr_df.columns)
    T_all = np.vstack([T_tr_df.values, T_te_df.values])
    np.save(TEXT_CACHE, T_all)
    pickle.dump(T_cols, open(TEXT_COLS_CACHE, 'wb'))
    lt.close(); del lm, tok; torch.cuda.empty_cache()

    # Correlation plot
    corr = T_tr_df.assign(y=y).corr()['y'].drop('y').sort_values()
    plt.figure(figsize=(7,7))
    corr.plot.barh(color=['#C44E52' if v<0 else '#4C72B0' for v in corr])
    plt.title('Pearson r of Text Features with Grammar Score')
    plt.xlabel('Pearson r'); plt.tight_layout(); plt.show()

Xt_tr = T_all[:len(train)]; Xt_te = T_all[len(train):]
print(f'\n Text features: {Xt_tr.shape}')

## 5. Tower D: Librosa Audio Features (Cached)
Handcrafted audio features capture speaking patterns that correlate with grammar proficiency: pauses, pitch stability, speaking rate, energy dynamics.

In [ ]:
AUDIO_CACHE = str(WORK / 'audio_features.npy')

if os.path.exists(AUDIO_CACHE):
    print('Loading cached audio features...')
    A_all = np.load(AUDIO_CACHE)
else:
    def audio_feats(path):
        try:
            ya, sr_ = librosa.load(str(path), sr=16000, mono=True)
            mfcc = librosa.feature.mfcc(y=ya, sr=sr_, n_mfcc=20)
            pitch, _ = librosa.piptrack(y=ya, sr=sr_)
            pv = pitch[pitch > 0]
            tempo, _ = librosa.beat.beat_track(y=ya, sr=sr_)
            rms = librosa.feature.rms(y=ya)[0]
            zcr = librosa.feature.zero_crossing_rate(ya)[0]
            spec = librosa.feature.spectral_centroid(y=ya, sr=sr_)[0]
            ethresh = np.percentile(rms, 10)
            return np.concatenate([
                mfcc.mean(1), mfcc.std(1),
                [pv.mean() if len(pv) else 0, pv.std() if len(pv) else 0,
                 float(tempo), rms.mean(), rms.std(), zcr.mean(), zcr.std(),
                 spec.mean(), spec.std(), len(ya)/sr_,
                 np.mean(rms < ethresh), np.percentile(rms, 25)],
            ])
        except Exception: return np.zeros(54)

    print('Extracting audio features...')
    all_paths = list(train.path) + list(test.path)
    A_all = np.stack([audio_feats(p) for p in tqdm(all_paths, desc='librosa')])
    np.save(AUDIO_CACHE, A_all)

A_tr = A_all[:len(train)]; A_te = A_all[len(train):]
print(f'\n Audio features: {A_tr.shape}')

## 6. Feature Assembly & Modelling with Repeated K-Fold CV
We combine all four feature towers and train an ensemble of regularised models.

In [ ]:
# Scale text and audio features
sc_txt = StandardScaler().fit(Xt_tr)
sc_aud = StandardScaler().fit(A_tr)
Xt_tr_s = sc_txt.transform(Xt_tr); Xt_te_s = sc_txt.transform(Xt_te)
A_tr_s  = sc_aud.transform(A_tr);  A_te_s  = sc_aud.transform(A_te)

X_tr = np.hstack([E_tr, W_tr, Xt_tr_s, A_tr_s])
X_te = np.hstack([E_te, W_te, Xt_te_s, A_te_s])
print(f'Full feature matrix: {X_tr.shape}')
print(f'  Whisper encoder:  {E_tr.shape[1]}')
print(f'  wav2vec2-base:    {W_tr.shape[1]}')
print(f'  Text features:    {Xt_tr.shape[1]}')
print(f'  Audio features:   {A_tr.shape[1]}')

def get_models():
    return {
        'ridge':     make_pipeline(StandardScaler(), Ridge(alpha=3000)),
        'ridge_pca': make_pipeline(StandardScaler(), PCA(128, random_state=SEED), Ridge(alpha=100)),
        'svr':       make_pipeline(StandardScaler(), PCA(128, random_state=SEED), SVR(C=2, epsilon=0.15, gamma='scale')),
        'krr':       make_pipeline(StandardScaler(), PCA(128, random_state=SEED), KernelRidge(alpha=1.0, kernel='rbf', gamma=1/400)),
        'elastic':   make_pipeline(StandardScaler(), PCA(128, random_state=SEED), ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000)),
        'svr_text':  make_pipeline(StandardScaler(), SVR(C=1, epsilon=0.2)),
        'lgbm':      lgb.LGBMRegressor(n_estimators=300, learning_rate=0.02, max_depth=4, num_leaves=15, subsample=0.7, colsample_bytree=0.3, random_state=SEED, verbose=-1),
    }

def feats_for(name, Xfull, Xtext):
    return Xtext if name == 'svr_text' else Xfull

rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)
names = list(get_models())
oof = {n: np.zeros(len(y)) for n in names}
cnt = np.zeros(len(y))

for tr_idx, va_idx in tqdm(list(rkf.split(X_tr)), desc='CV'):
    for n, m in get_models().items():
        Xa = feats_for(n, X_tr, Xt_tr_s)
        m.fit(Xa[tr_idx], y[tr_idx])
        oof[n][va_idx] += m.predict(Xa[va_idx])
    cnt[va_idx] += 1

oof = {n: v / cnt for n, v in oof.items()}
res = pd.DataFrame({n: {'CV_RMSE': rmse(y, np.clip(v,0,5)), 'CV_Pearson': pearsonr(y,v)[0]} for n,v in oof.items()}).T.sort_values('CV_RMSE')
print('\nPer-model CV results:'); print(res.to_string())

In [ ]:
# Optimised stacking blend
O = np.column_stack([oof[n] for n in names])
w_nnls, _ = nnls(O, y); w_nnls = w_nnls / (w_nnls.sum() + 1e-8)
blend_nnls = np.clip(O @ w_nnls, 0, 5)

def blend_rmse(weights):
    weights = np.abs(weights); weights = weights / (weights.sum() + 1e-8)
    return rmse(y, np.clip(O @ weights, 0, 5))

result = minimize(blend_rmse, w_nnls, method='Nelder-Mead', options={'maxiter': 10000, 'xatol': 1e-6})
w_nm = np.abs(result.x); w_nm = w_nm / (w_nm.sum() + 1e-8)
blend_nm = np.clip(O @ w_nm, 0, 5)

if rmse(y, blend_nm) < rmse(y, blend_nnls):
    w, blend = w_nm, blend_nm; print('Using Nelder-Mead weights.')
else:
    w, blend = w_nnls, blend_nnls; print('Using NNLS weights.')

print('\nBlend weights:')
for n, wi in zip(names, w): print(f'  {n:15s}: {wi:.4f}')
print(f'\nBLEND CV RMSE    = {rmse(y, blend):.4f}')
print(f'BLEND CV Pearson = {pearsonr(y, blend)[0]:.4f}')
print(f'Baseline RMSE    = {np.std(y):.4f}')

## 7. Final Fit, Training RMSE (Required), Visualisations & Submission

In [ ]:
from IPython.display import FileLink, display

P_tr = []; P_te = []
for n, m in get_models().items():
    Xa = feats_for(n, X_tr, Xt_tr_s); Xb = feats_for(n, X_te, Xt_te_s)
    m.fit(Xa, y); P_tr.append(m.predict(Xa)); P_te.append(m.predict(Xb))

train_pred = np.clip(np.column_stack(P_tr) @ w, 0, 5)
test_pred  = np.clip(np.column_stack(P_te) @ w, 0, 5)

print('='*60)
print(f'TRAINING RMSE (in-sample, full fit)  : {rmse(y, train_pred):.4f}')
print(f'TRAINING Pearson                     : {pearsonr(y, train_pred)[0]:.4f}')
print(f'CROSS-VALIDATED RMSE (honest)        : {rmse(y, blend):.4f}')
print(f'CROSS-VALIDATED Pearson (honest)     : {pearsonr(y, blend)[0]:.4f}')
print('='*60)

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].scatter(y, blend, alpha=0.4, s=15, c='#4C72B0')
ax[0].plot([0,5],[0,5],'r--',lw=1.5)
ax[0].set(xlabel='True Score', ylabel='OOF Prediction', title='OOF Predicted vs True')
sns.histplot(y-blend, kde=True, ax=ax[1], color='#55A868', bins=30)
ax[1].set_title('OOF Residuals'); ax[1].set_xlabel('Error')
sns.kdeplot(y, label='Train True', ax=ax[2], color='#4C72B0')
sns.kdeplot(test_pred, label='Test Pred', ax=ax[2], color='#C44E52')
ax[2].legend(); ax[2].set_title('Score Distributions')
model_rmses = {n: rmse(y, np.clip(oof[n],0,5)) for n in names}
pd.Series(model_rmses).sort_values().plot.barh(ax=ax[3], color='#DD8452')
ax[3].set_title('Per-Model CV RMSE'); ax[3].set_xlabel('RMSE')
plt.tight_layout(); plt.show()

# ── Save submission ────────────────────────────────────────────────────────────
out = pd.DataFrame({sub.columns[0]: test[FCOL].values, sub.columns[1]: test_pred})
out.to_csv('/kaggle/working/submission.csv', index=False)
print(f'\nsubmission.csv saved ({len(out)} rows)')
print(out.head(10))
print(f'\nTest preds: min={test_pred.min():.3f} max={test_pred.max():.3f} mean={test_pred.mean():.3f}')

# ── Auto-download link ─────────────────────────────────────────────────────────
print('\n' + '='*60)
print('CLICK THE LINK BELOW TO DOWNLOAD YOUR SUBMISSION:')
print('='*60)
display(FileLink('/kaggle/working/submission.csv', result_html_prefix='📥 Download: '))


## 8. Discussion

### What worked
- **Whisper-large-v3 encoder embeddings** carry the strongest signal — they capture both what was said and how it was said (hesitation, fluency, pronunciation) in a high-dimensional space.
- **wav2vec2-base embeddings** add a complementary acoustic view trained purely on speech, capturing pronunciation quality and rhythm.
- **GPT-2 perplexity** acts as a grammar proxy: ungrammatical or disfluent transcripts receive higher perplexity scores.
- **LanguageTool grammar errors** directly count rule-based grammar and spelling mistakes.
- **Librosa audio features** (MFCCs, pitch, energy, pause ratio) capture speaking patterns correlated with proficiency.
- **Heavy regularisation** (Ridge alpha=3000, PCA, SVR) is critical with only 769 training samples.
- **Stacking blend** with Nelder-Mead optimisation finds the optimal model combination.

### Honest metrics
- The **training RMSE** is optimistic (in-sample); the **cross-validated RMSE** is the number to trust.
- We use RepeatedKFold (5x3) to reduce variance in the CV estimate.

### Limitations & future work
- Whisper auto-corrects grammar in transcripts, limiting text-only features. **Direct encoder hidden states** partially mitigate this.
- Fine-tuning wav2vec2-base end-to-end as a regression model could further improve performance.
- With more compute, larger models (WavLM-Large, HuBERT-XL) or multi-seed bagging could reduce variance.